In [1]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from training.train_and_evaluate_relation_extraction import load_stored_dataset_combination_graph
from custom_datasets.dataframe_dataset import DFDataset
from pipeline.pipeline import *
from flair.embeddings import TransformerWordEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer
from flair.data import Sentence

/workspace/llm-graph-construction


/workspace/llm-graph-construction/graph_building/llm/OpenChat.py:10: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama = Ollama(base_url='http://localhost:11434',


In [3]:
dataset_train, dataset_val, dataset_test_ub = load_stored_dataset_combination_graph(balanced=True, dataset="i2b2")
number_of_relations = 3

In [4]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
test_name = "i2b2"
event_extractor = "bert"

In [6]:
if event_extractor == "flair":
    from flair.models import SequenceTagger
    from flair.embeddings import TransformerWordEmbeddings
    from flair.models import SequenceTagger
    from flair.trainers import ModelTrainer
    from flair.data import Sentence
    def get_predictions(tagged_sentence, original_text):
        predicted_events = []
        curent_event = [0,0]
        in_event = False
        offset = 0
        for word in tagged_sentence:
            offset = original_text.find(word.text, offset)
            if word.tag == "B":
                if in_event:
                    # complete curent event
                    predicted_events.append((curent_event[0], curent_event[1], original_text[curent_event[0]: curent_event[1]]))
                    current_event = [offset, offset]
                    in_event = False
                in_event = True
                curent_event[0] = offset
                curent_event[1] = offset + len(word.text)
            elif word.tag == "I" and in_event == True:
                curent_event[1] = offset + len(word.text)
            elif word.tag == "I" and in_event == False:
                # ignore events without a beginning
                # in_event = True
                # curent_event[0] = offset
                # curent_event[1] = offset + len(word.text)
                pass
            elif word.tag == "X" and in_event == True:
                # complete curent event
                predicted_events.append((curent_event[0], curent_event[1], original_text[curent_event[0]: curent_event[1]]))
                curent_event = [offset, offset]
                in_event = False
            
            
            offset += len(word.text)
        return predicted_events

    if test_name == "thyme":
        model = SequenceTagger.load('resources/taggers/sota-ner-flert-'+test_name+'/final-model.pt')
    else:
        model = SequenceTagger.load('resources/taggers/sota-ner-flert-'+test_name+'2/final-model.pt')
    def extract_events(text):
        sentence = Sentence(text)
        # predict tags and print
        model.predict(sentence)
        predictions = get_predictions(sentence, text)
        return predictions

        
elif event_extractor == "spert":
    import nltk
    import json
    nltk.download('punkt_tab')
    def split_document_if_too_long(text):
        max_length = 700 # 700 characters
        def sentence_spans(txt):
            tokens=nltk.sent_tokenize(txt)
            offset = 0
            for token in tokens:
                offset = txt.find(token, offset)
                yield token, offset, offset+len(token)
                offset += len(token)
        sentences = sentence_spans(text)
    
        documents = []
        start_offset = 0
        end_offset = 0
        document_offsets = []
        document_token_spans = []
        for sentence in sentences:
            new_document_length = sentence[2] - start_offset
            if end_offset - start_offset > 0 and new_document_length > max_length:
                doc_text = text[start_offset: end_offset]
                documents.append({"tokens": nltk.word_tokenize(doc_text)})
                document_offsets.append(start_offset)
                document_token_spans.append(list(spans(doc_text)))
                start_offset = end_offset
            end_offset = sentence[2]
        if end_offset - start_offset > 0:
            doc_text = text[start_offset: end_offset]
            documents.append({"tokens": nltk.word_tokenize(doc_text)})
            document_offsets.append(start_offset)
            document_token_spans.append(list(spans(doc_text)))
        return documents, document_offsets, document_token_spans
    def convert_text(text):
        tokens=nltk.word_tokenize(text)
        # Serializing json
        json_object, document_offsets, document_token_spans = split_document_if_too_long(text)
         
        # Writing to sample.json
        with open("/workspace/spert/data/predict.json", "w") as outfile:
            outfile.write(json.dumps(json_object))
        return document_offsets, document_token_spans
    def prepare_config():
        with open("/workspace/spert/configs/predict.conf", "w") as outfile:
            outfile.write("[1]" + "\n")
            outfile.write("model_type = spert" + "\n")
            outfile.write("model_path = data/models/" + test_name + "\n")
            outfile.write("tokenizer_path = data/models/" + test_name + "\n")
            outfile.write("dataset_path = data/predict.json" + "\n")
            outfile.write("types_path = data/datasets/i2b2/i2b2_types.json" + "\n")
            outfile.write("predictions_path = data/predictions.json" + "\n")
            outfile.write("spacy_model = en_core_web_sm" + "\n")
            outfile.write("eval_batch_size = 1" + "\n")
            outfile.write("rel_filter_threshold = 0.4" + "\n")
            outfile.write("size_embedding = 25" + "\n")
            outfile.write("prop_drop = 0.1" + "\n")
            outfile.write("max_span_size = 10" + "\n")
            outfile.write("sampling_processes = 4" + "\n")
            outfile.write("max_pairs = 1000" + "\n")

    def spans(txt):
        tokens=nltk.word_tokenize(txt)
        offset = 0
        for token in tokens:
            offset = txt.find(token, offset)
            yield token, offset, offset+len(token)
            offset += len(token)
            
    def extract_events(text):
        document_offsets, document_token_spans = convert_text(text)
        prepare_config()
        
        bash_command = 'bash -c "cd /workspace/spert; python spert.py predict --config configs/predict.conf"'
        os.system(bash_command)

        with open('/workspace/spert/data/predictions.json', 'r') as file:
            predictions = json.load(file)
        predicted_events = []
        print(predictions)
        for i, doc_predictions in enumerate(predictions):
            for pred in doc_predictions["entities"]:
                char_start = document_token_spans[i][pred["start"]][1] + document_offsets[i]
                # print(pred["end"], len(document_token_spans[i]))
                char_end = document_token_spans[i][pred["end"]-1][2] + document_offsets[i]
                if (pred["start"] >= pred["end"]):
                    print("End ni po start:", pred["start"], pred["end"])
                # print(char_start, char_end, i)
                predicted_events.append((char_start, char_end, text[char_start:char_end]))
        return predicted_events
    pass

elif event_extractor == "bert":
    def split_text_into_sentenes(text):
        tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
        sentences = list(tokenizer.span_tokenize(text))
        return sentences
    def extract_events(text):
        event_extraction_model = torch.load("event-model.pt", map_location=torch.device('cpu'))
        sentence_idxs = split_text_into_sentenes(text)
        events = []
        for sentence_idx in sentence_idxs:
            tokenized = event_extraction_model.tokenizer(text[sentence_idx[0]:sentence_idx[1]], return_tensors="pt")
            classification = event_extraction_model(tokenized)
            predictions = torch.argmax(classification["results"], axis=1)
            start = 0
            end = 0
            inside_event = False
            for i, prediction in enumerate(predictions):
                if prediction == 1:
                    if tokenized.token_to_chars(i) is None:
                        continue
                    if not inside_event:
                        start = tokenized.token_to_chars(i).start
                    end = tokenized.token_to_chars(i).end
                    inside_event = True
                else:
                    if inside_event:
                        # end this event
                        events.append((sentence_idx[0] + start, sentence_idx[0] + end, text[sentence_idx[0] + start:sentence_idx[0] + end]))
                    inside_event = False
            if inside_event:
                # end this event
                events.append((sentence_idx[0] + start, sentence_idx[0] + end, text[sentence_idx[0] + start:sentence_idx[0] + end]))
        return events


In [7]:
dataframe = dataset_test_ub.df

documents = set(dataframe["document_id"])
dataset = []
for doc in documents:
    relations = dataframe[dataframe["document_id"] == doc]
    text = relations["text"].iloc[0]
    graph = []
    for row in relations.iloc:
        graph.append(((row["event1_start"], row["event1_end"], row["event1_text"]), row["class"], (row["event2_start"], row["event2_end"], row["event2_text"])))
    dataset.append({"text": text, "graph": graph})

In [8]:
def overlap(event1, event2):
    start1, end1, text1 = event1
    start2, end2, text2 = event2
    if start1 >= start2 and start1 <= end2:
        # print(event1, event2)
        return True
    if end1 >= start2 and end1 <= end2:
        # print(event1, event2)
        return True
    if start2 >= start1 and start2 <= end1:
        # print(event1, event2)
        return True
    return False

In [9]:
def most_simmilar_event(event_list, event_target):
    # min_difference = -1
    # best_match = None
    for e in event_list:
        if overlap(e, event_target):
            return e
        # dif = -1
        # if e in event_target:
        #     dif = len(event_target) - len(e)
        # if event_target in e:
        #     dif = len(e) - len(event_target)
        # if dif >= 0 and (min_difference < 0 or min_difference > dif):
        #     min_difference = dif
        #     best_match = e
    return None

def find_relation(graph, event1, event2, expected_relation):
    for ind, r in enumerate(graph):
        if r[0] == event1 and r[2] == event2 and r[1] == expected_relation:
            return ind, r
    for ind, r in enumerate(graph):
        if r[0] == event1 and r[2] == event2:
            return ind, r
    return -1, None

# compare_graphs(truth, prediction)
def compare_graphs(graph1, graph2):
    event_map = {}
    events1 = list(set([r[0] for r in graph1]+[r[2] for r in graph1]))
    events2 = list(set([r[0] for r in graph2]+[r[2] for r in graph2]))
    for e in events1:
        event_map[e] = most_simmilar_event(events2, e)
    #print(event_map)
    
    matching_relation = [False for _ in range(len(graph2))]
    
    correct = 0
    incorrect = 0
    missing = 0
    too_much = 0
    for relation in graph1:
        ind2, relation2 = find_relation(graph2, event_map[relation[0]], event_map[relation[2]], relation[1])
        if relation2 is None:
            missing += 1
        else:
            matching_relation[ind2] = True
            if relation[1] == relation2[1]:
                correct += 1
            else:
                incorrect += 1
    too_much = len(matching_relation) - sum(matching_relation)
    #print(correct, incorrect, missing, too_much)
    return correct, incorrect, missing, too_much


In [10]:
def get_event_pairs_of_interest(example):
    graph = example["graph"]
    event_pairs = []
    for g in graph:
        event_pairs.append((g[0], g[2]))
    return list(event_pairs)

from custom_datasets.combining_data import window_row_entity_bert
def window_text(graph):
    graph = window_row_entity_bert(graph, normalize_event_order=False)
    if graph is None:
        return None
    return graph

In [11]:
def analyze_document(text, patient_id, event_pairs_of_interest=None):
    print(len(event_pairs_of_interest))
    events = extract_events(text)
    # print("Events:")
    # print(events)
    if event_pairs_of_interest is None:
        event_pairs = generate_event_pairs(text, events)
    else:
        event_pairs = []
        for e1, e2 in event_pairs_of_interest:
            e1 = most_simmilar_event(events, e1)
            e2 = most_simmilar_event(events, e2)
            if e1 is not None and e2 is not None:
                event_pairs.append((e1, e2))
    print(len(event_pairs))
    # print("Pairs ("+str(len(event_pairs))+"):")
    # print(event_pairs)
    dataframe = construct_basic_dataframe(text, event_pairs, 0)
    print(len(dataframe))
    # print("Dataframe")
    # print(dataframe)
    dataset = construct_dataset_with_graphs(text, dataframe, patient_id)
    print(len(dataset))
    # Window text into the same format as when training the model (windowing is already performed in convert_row_to_graph function
    # dataset.generated = list(filter(lambda x: x is not None, map(window_text, dataset.generated)))
    relations = predict_temporal_relations(dataset)
    return relations

In [12]:
def predict_temporal_relations(dataset):
    human_readable_predictions = []
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = torch.load("best-models/bimodal-model-i2b2.pt", map_location=device)

    loader = DataLoader(dataset, batch_size=16)
    for batch in loader:
        batch.to(device)
        labels = batch.y
        result = model(data=batch, labels=labels)
        predictions = np.argmax(result["predictions"].cpu().detach().numpy(), axis=1)
        for i in range(len(batch["text"])):
            event1 = batch["event1_oroginal_position"][i]
            event2 = batch["event2_oroginal_position"][i]
            human_readable_predictions.append((event1, relation_types[predictions[i]], event2))
    return human_readable_predictions


In [13]:
transitivity = {
    ("OVERLAP", "OVERLAP"): "OVERLAP",
    ("BEFORE", "OVERLAP"): "BEFORE",
    ("BEFORE", "BEFORE"): "BEFORE",
    ("OVERLAP", "BEFORE"): "BEFORE",
    ("AFTER", "OVERLAP"): "AFTER",
    ("OVERLAP", "AFTER"): "AFTER",
    ("AFTER", "AFTER"): "AFTER",
    ("AFTER", "BEFORE"): "OVERLAP",
    ("BEFORE", "AFTER"): "OVERLAP"
}

inverse = {
    "BEFORE": "AFTER",
    "AFTER": "BEFORE",
    "OVERLAP": "OVERLAP"
}

def does_relation_exist(graph, event1, event2):
    ind, relation = find_relation(graph, event1, event2, None)
    return ind >= 0

def add_transitive(graph):
    for i in range(len(graph)):
        for j in range(i+1, len(graph)):
            if graph[i][2] == graph[j][0]:
                # matching relations
                event1 = graph[i][0]
                event2 = graph[j][2]
                if not does_relation_exist(graph, event1, event2):
                    relation = transitivity[(graph[i][1], graph[j][1])]
                    new_relation = (event1, relation, event2)
                    graph.append(new_relation)
    return graph

def add_inverse(graph):
    for i in range(len(graph)):
        event1 = graph[i][2]
        event2 = graph[i][0]
        if not does_relation_exist(graph, event1, event2):
            relation = inverse[graph[i][1]]
            new_relation = (event1, relation, event2)
            graph.append(new_relation)
    return graph

def graph_closure(graph):
    graph = add_inverse(graph)
    graph = add_transitive(graph)
    return graph

In [ ]:
for patient_ind, example in enumerate(dataset):
    print(str(patient_ind) + "/" + str(len(dataset)))
    event_pairs_of_interest = get_event_pairs_of_interest(example)
    graph_predicted = analyze_document(example["text"], patient_ind, event_pairs_of_interest=event_pairs_of_interest)
    break
    

0/120
44
36
36
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmpvdeq7gls



00%|█████████████████████████████████████████| 492M/492M [01:07<00:00, 7.68MiB/s]

Finished download, copying /tmp/tmpvdeq7gls to cache at /root/.scispacy/datasets/2b79923846fb52e62d686f2db846392575c8eb5b732d9d26cd3ca9378c622d40.87bd52d0f0ee055c1e455ef54ba45149d188552f07991b765da256a1b512ca0b.tfidf_vectors_sparse.npz
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/nmslib_index.bin not found in cache, downloading to /tmp/tmpuyoug6a9



00%|█████████████████████████████████████████| 724M/724M [01:33<00:00, 8.13MiB/s]

Finished download, copying /tmp/tmpuyoug6a9 to cache at /root/.scispacy/datasets/7e8e091ec80370b87b1652f461eae9d926e543a403a69c1f0968f71157322c25.6d801a1e14867953e36258b0e19a23723ae84b0abd2a723bdd3574c3e0c873b4.nmslib_index.bin
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectorizer.joblib not found in cache, downloading to /tmp/tmpecki_nh_


...

Finished download, copying /tmp/tmpecki_nh_ to cache at /root/.scispacy/datasets/37bc06bb7ce30de7251db5f5cbac788998e33b3984410caed2d0083187e01d38.f0994c1b61cc70d0eb96dea4947dddcb37460fb5ae60975013711228c8fe3fba.tfidf_vectorizer.joblib
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/concept_aliases.json not found in cache, downloading to /tmp/tmp298rhhs1



00%|█████████████████████████████████████████| 264M/264M [00:36<00:00, 7.61MiB/s]

Finished download, copying /tmp/tmp298rhhs1 to cache at /root/.scispacy/datasets/6238f505f56aca33290aab44097f67dd1b88880e3be6d6dcce65e56e9255b7d4.d7f77b1629001b40f1b1bc951f3a890ff2d516fb8fbae3111b236b31b33d6dcf.concept_aliases.json
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/kbs/2023-04-23/umls_2022_ab_cat0129.jsonl not found in cache, downloading to /tmp/tmph_1ohbo0



00%|█████████████████████████████████████████| 628M/628M [01:24<00:00, 7.79MiB/s]

Finished download, copying /tmp/tmph_1ohbo0 to cache at /root/.scispacy/datasets/d5e593bc2d8adeee7754be423cd64f5d331ebf26272074a2575616be55697632.0660f30a60ad00fffd8bbf084a18eb3f462fd192ac5563bf50940fc32a850a3c.umls_2022_ab_cat0129.jsonl
